# modernCNN

### Imports

In [1]:
# Добавляем текущую директорию в путь
import os
import sys
sys.path.append(os.path.abspath('..'))

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets
from pathlib import Path
import logging
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from src.models import load_model, confusion_matrix, Trainer
from src.data import load_datasets, create_dataloaders
from src.utils import utils

%matplotlib inline

In [3]:
DIR = Path.cwd().parent

### Logging

In [4]:
# Настройка логгера
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

### Loading the configuration for the dataset

In [5]:
path_config_data = Path("../configs/data.yaml").absolute()
config_data = utils.load_config(path_config_data)

In [6]:
config_data["image_size"]

128

In [7]:
# Путь хранения датасета
path_data_load = Path(DIR, config_data["root_dir"])
train_dataset, val_dataset = load_datasets(path_data_load, config_data["image_size"])

In [8]:
# Создаем DataLoader
train_loader, val_loader = create_dataloaders(
        train_dataset,
        val_dataset,
        config_data['batch_size'],
        config_data['num_workers'])

In [9]:
# for batch_idx, (data, target) in enumerate(train_loader):
#     print(f"Target range: {target.min().item()} - {target.max().item()}")
#     print(f"Unique targets: {torch.unique(target)}")
#     print(f"Number of classes in batch: {len(torch.unique(target))}")
#     break

### Loading the configuration for the model

In [10]:
path_config_model = Path("../configs/modernCNN.yaml").absolute()
config_dict = utils.load_config(path_config_model)
config_model = config_dict['model']

In [11]:
# Фиксируем random seed для воспроизводимости
utils.set_seed(config_model['seed'])

In [12]:
# Определение устройства
device_obj = torch.device(
    config_model['device'] if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device_obj}")

Using device: cuda


### Model ModernCNN

In [13]:
model = load_model(config_model['name'], config_data['num_classes'])
model = model.to(device_obj)

In [14]:
model.summary((3, config_data["image_size"], config_data["image_size"]))


        Model: ModernCNN
        Input shape:  (3, 128, 128)
        Output shape: (1, 37)
        
        Total parameters: 2,806,117
        Trainable parameters: 2,806,117
        
        Architecture:
        - Stem: Conv3x3-BN-ReLU
        - Stage 1: 2x Residual blocks, 64 channels
        - Stage 2: 2x Residual blocks, 128 channels
        - Stage 3: 4x Residual blocks, 256 channels
        - Classifier: AdaptiveAvgPool + Dropout + Linear
        
        Features:
        - ReLU activation
        - SE blocks for channel attention
        - Residual connections
        - Kaiming initialization
        


### Optimizer

In [15]:
# Оптимизатор и scheduler
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config_model['lr'],
    weight_decay=config_model['weight_decay'],
)

In [16]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, eta_min=1e-4, T_max=10)

### Train model

In [17]:
# Создание тренера
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device_obj,
    output_dir=Path(DIR, config_model['output_dir']),
    logger=logger,
)

### Epochs

In [ ]:
# Устанавливаем начальную эпоку если ранее было обучение модель продолжит
# с того места на котором остановилось обучение
start_epoch:int = 0
checkpoint_path = Path(DIR, config_dict['resume'])
if config_dict.get('resume') and checkpoint_path.exists():
    checkpoint = trainer.load_checkpoint(checkpoint_path)
    start_epoch = checkpoint.get('epoch', 0) + 1
    print(f"Resumed from epoch {start_epoch}")

In [ ]:
best_metric = trainer.train(
    num_epochs=config_model['epochs'],
    start_epoch=start_epoch,
)

In [ ]:
best_metric

### best model checkpoint

In [18]:
best_model_checkpoint = trainer.load_checkpoint(Path(DIR, "./data/models/exp_modern_cnn/best_model.pth"))
best_model_checkpoint['metric']

0.3041700735895339

### Confusion matrix

In [ ]:
# confusion_matrix(trainer.model, val_loader, config_model['device'], config_model['num_classes'], val_dataset.classes)

Моедль часто путает класс American Pit Bull Terrier с классом Staffordshire Bull Terrier.
Собаки похожи друг на друга  и имеют минимальные отличия.

### Resume

Моедль нужно дообучать и подбирать параметры. Увеличивать количество параметров.